# 6주차 3차시: Pandas 심화 & RAG 기반 ETF 검색

| 주제 | 내용 |
|---|---|
| Pandas 심화 | Long-format · `groupby` · `pivot_table` · `stack/unstack` |
| Document 구성 | LLM 생성 description + 메타데이터 |
| Vector 검색 | FAISS similarity search |
| BM25 검색 | Kiwi 형태소 분석 + `BM25Okapi` |
| 메타 필터 | `filtered_search` — 카테고리·보수율 등 조건 결합 |


In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.0/513.0 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
!pip install finance-datareader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 2.0 MB/s eta 0:00:00


In [ ]:
import os, json, math
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# from dotenv import load_dotenv
# load_dotenv()
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
MODEL = "gpt-4o-mini"

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
import FinanceDataReader as fdr

## 전일 실습 이어서 — Pandas 심화
- Pandas의 데이터 프레임 사용법 확인하는중 (기능 및 데이터 분석에 활용)

In [ ]:
# meta와 price_dict를 딕셔너리 형태로 입력받을거임
kodex = fdr.DataReader('069500', start=(datetime.now() - timedelta(days=180)).strftime('%Y-%m-%d'))
tiger = fdr.DataReader('360750', start=(datetime.now() - timedelta(days=180)).strftime('%Y-%m-%d'))
bond = fdr.DataReader('152380', start=(datetime.now() - timedelta(days=180)).strftime('%Y-%m-%d'))
price_dict = {
    '069500': kodex,
    '360750': tiger,
    '152380': bond,
}
meta = pd.DataFrame([
    {'ticker': '069500', 'name': 'KODEX 200', 'category': '국내주식', 'expense': 0.15},
    {'ticker': '360750', 'name': 'TIGER S&P500', 'category': '해외주식', 'expense': 0.07},
    {'ticker': '152380', 'name': 'KODEX 국고채10년', 'category': '채권', 'expense': 0.05},
    {'ticker': '132030', 'name': 'KODEX 골드선물', 'category': '원자재', 'expense': 0.68},
])


def merge_all(price_dict, meta_df):
  # 1. price_dict에 있는 ticker 들의 close 들
  closes = pd.concat({ticker: df['Close'] for ticker, df in price_dict.items()}, axis=1).dropna() # axis=1은 옆으로 붙여라(column이 추가되는 방향)

  # 2. price_dict에 있는 close들의 return, annual return
  analysis_rows = []
  for ticker in closes.columns:
    ret = closes[ticker].pct_change()
    ann_ret = (1 + ret.mean()) ** 255 -1
    analysis_rows.append({
        'ticker' : ticker,
        'annual_return' : round(ann_ret * 100, 2)


    })
    analysis_df = pd.DataFrame(analysis_rows)

  # 3. 1과 2를 merge한 dataframe을 만들어서, 반환
  result = pd.merge(meta_df, analysis_df, on = 'ticker', how= 'inner')

  return result # 3

### Long-format 데이터 구성 & `groupby` 집계

- 여러 종목 시계열을 `[{ticker, name, date, return, month}, ...]` 리스트로 쌓아 long-format DataFrame 구성
- `groupby(col).agg([...])` 로 종목별 평균 수익률·관측수 집계
- wide(종목별 열) ↔ long(행 누적) 전환이 시각화·머신러닝 입력 시 기초


In [ ]:
long_data = []
for ticker, name in [('069500', 'KODEX200'), ('360750', 'TIGER_SP500'), ('152380', 'KODEX_국고채')]:
    df = price_dict.get(ticker)
    if df is None:
        continue
    ret = df['Close'].pct_change().dropna()
    for date, val in ret.items():
        long_data.append({
            'date': date,
            'ticker': ticker,
            'name': name,
            'return': val,
            'weekday': date.strftime('%a'),
            'month': date.month,
        })
long_df = pd.DataFrame(long_data)
print(f"Long format: {long_df.shape}")

Long format: (363, 6)


In [ ]:
long_df.head()

,date,ticker,name,return,weekday,month
0,2025-10-21,069500,KODEX200,0.002149,Tue,10
1,2025-10-22,069500,KODEX200,0.011338,Wed,10
2,2025-10-23,069500,KODEX200,-0.010565,Thu,10
3,2025-10-24,069500,KODEX200,0.026108,Fri,10
4,2025-10-27,069500,KODEX200,0.027496,Mon,10


In [ ]:
long_df.tail()

,date,ticker,name,return,weekday,month
358,2026-04-10,152380,KODEX_국고채,-0.001933,Fri,4
359,2026-04-13,152380,KODEX_국고채,-0.002533,Mon,4
360,2026-04-14,152380,KODEX_국고채,0.005377,Tue,4
361,2026-04-15,152380,KODEX_국고채,0.001486,Wed,4
362,2026-04-16,152380,KODEX_국고채,-0.003041,Thu,4


In [ ]:
# aggregate: 모으다
# name을 기준으로 group이 되었고 아래 속성에 따라서 column 등, 요약 및 통계된 결과를 확인가능
summary = long_df.groupby('name')['return'].agg(
    평균수익률 = 'mean',
    변동성 = 'std',
    최대 = 'max',
    최소 = 'min',
    거래일수 = 'count'
)
summary

,평균수익률,변동성,최대,최소,거래일수
name,,,,,
KODEX200,0.005201,0.030707,0.101054,-0.124618,121
KODEX_국고채,-0.000356,0.004471,0.017152,-0.012295,121
TIGER_SP500,0.000806,0.008249,0.018928,-0.026759,121


In [ ]:
summary['평균수익률'] = (summary['평균수익률']*100).round(4)

In [ ]:
summary

,평균수익률,변동성,최대,최소,거래일수
name,,,,,
KODEX200,0.5201,0.030707,0.101054,-0.124618,121
KODEX_국고채,-0.0356,0.004471,0.017152,-0.012295,121
TIGER_SP500,0.0806,0.008249,0.018928,-0.026759,121


### Pivot Table & stack / unstack

- `pd.pivot_table(df, values, index, columns, aggfunc)` — groupby + reshape 를 한 번에
- `df.stack()` — 컬럼을 인덱스로 내림 (wide → long)
- `stacked.idxmax()` / `.max()` 로 전체 행렬에서 최댓값의 (row, col) 라벨·값 조회


In [ ]:
pd.pivot_table(long_df, values = 'return', index = 'name', columns='month', aggfunc='mean')
# return 컬럼을 활용해 pivot table을 만들었고 index 컬럼은 'name'을 활용함
# columns, 즉 열들은 month로 지정함
# aggfunc는, 미리 지정된 통계함수를 넣음

month,1,2,3,4,10,11,12
name,,,,,,,
KODEX200,0.011545,0.012140,-0.009222,0.019771,0.009240,-0.001922,0.004390
KODEX_국고채,-0.000733,0.001072,-0.001459,0.001687,-0.001515,-0.001065,-0.000026
TIGER_SP500,0.000011,-0.000227,-0.000399,0.004794,0.003455,0.001442,-0.000375


In [ ]:
vol_pivot = pd.pivot_table(long_df, values = 'return', index = 'name', columns = 'month', aggfunc = 'std')
# volatility pivot

In [ ]:
vol_pivot

month,1,2,3,4,10,11,12
name,,,,,,,
KODEX200,0.011950,0.032510,0.051126,0.037253,0.014190,0.023680,0.013825
KODEX_국고채,0.003706,0.003513,0.005920,0.007576,0.002494,0.003578,0.002856
TIGER_SP500,0.007987,0.008705,0.007879,0.004632,0.006583,0.011211,0.007453


In [ ]:
stacked = vol_pivot.stack()
stacked

name         month
KODEX200     1        0.011950
             2        0.032510
             3        0.051126
             4        0.037253
             10       0.014190
             11       0.023680
             12       0.013825
KODEX_국고채    1        0.003706
             2        0.003513
             3        0.005920
             4        0.007576
             10       0.002494
             11       0.003578
             12       0.002856
TIGER_SP500  1        0.007987
             2        0.008705
             3        0.007879
             4        0.004632
             10       0.006583
             11       0.011211
             12       0.007453
dtype: float64

In [ ]:
stacked.idxmax()

('KODEX200', np.int64(3))

In [ ]:
stacked.max()

0.05112596631250477

---

### 여기까지가 Pandas 활용한 데이터 분석, 전처리 등 내용을 다룬 부분
---

ㅇ

## RAG 기반 ETF 추천 — 개요

사용자의 자연어 쿼리에 맞는 ETF 를 찾기 위한 Retriever 파이프라인.

```
데이터 수집 → 지표 계산 → LLM Description 생성 → Document 구성 → Embedding/BM25 인덱싱 → Retriever(Vector + BM25 + Meta Filter)
```

- Document = `page_content`(설명 텍스트) + `metadata`(구조화된 조건)
- 의미(Vector) · 키워드(BM25) · 조건(Meta Filter) 3축 결합이 다음 차시 하이브리드 검색의 기반


In [ ]:
import os, json, time, re, math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from collections import Counter
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS as LangchainFAISS
from langchain_core.documents import Document

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
MODEL = "gpt-4o-mini"
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [ ]:
# 1. 데이터 생성 필요
# 2. ETF 통계정보 등도 필요

### 1. 데이터 수집 (원문: "1, 데이터 생성" — 콤마 오탈자 및 셀 21 주석의 "1. 데이터 수집" 과 통일)

- `fdr.StockListing("ETF/KR")` 로 국내 ETF 전체 목록 수집
- `tickers_info` 딕셔너리에 카테고리·보수율·배당률·키워드 등 메타정보 수동 정리


In [ ]:
import FinanceDataReader as fdr
etf_listing = fdr.StockListing('ETF/KR')

In [ ]:
etf_listing.head()

,Symbol,Category,Name,Price,RiseFall,Change,ChangeRate,NAV,EarningRate,Volume,Amount,MarCap
0,069500,1,KODEX 200,94710,2,2045,2.21,94689.0,32.4893,15401829,1449982,218496
1,360750,4,TIGER 미국S&P500,25780,2,265,1.04,25695.0,0.8957,17095776,439893,159784
2,396500,2,TIGER 반도체TOP10,35935,2,445,1.25,36079.0,48.8674,13968697,498769,96809
3,133690,4,TIGER 미국나스닥100,171995,2,3245,1.92,171121.0,1.4622,686569,117768,87029
4,379800,4,KODEX 미국S&P500,23560,2,245,1.05,23485.0,0.9205,22846526,537199,84404


In [ ]:
etf_listing['Symbol'].astype(str).str.zfill(6)

,Symbol
0,069500
1,360750
2,396500
3,133690
4,379800
...,...
1088,334700
1089,253160
1090,465620
1091,306520


In [ ]:
name_by_ticker = dict(zip(etf_listing['Symbol'].astype(str).str.zfill(6), etf_listing['Name']))
name_by_ticker

{'069500': 'KODEX 200',
 '360750': 'TIGER 미국S&P500',
 '396500': 'TIGER 반도체TOP10',
 '133690': 'TIGER 미국나스닥100',
 '379800': 'KODEX 미국S&P500',
 '102110': 'TIGER 200',
 '459580': 'KODEX CD금리액티브(합성)',
 '488770': 'KODEX 머니마켓액티브',
 '122630': 'KODEX 레버리지',
 '229200': 'KODEX 코스닥150',
 '278530': 'KODEX 200TR',
 '379810': 'KODEX 미국나스닥100',
 '411060': 'ACE KRX금현물',
 '310970': 'TIGER MSCI Korea TR',
 '357870': 'TIGER CD금리투자KIS(합성)',
 '091160': 'KODEX 반도체',
 '233740': 'KODEX 코스닥150레버리지',
 '498400': 'KODEX 200타겟위클리커버드콜',
 '423160': 'KODEX KOFR금리액티브(합성)',
 '381180': 'TIGER 미국필라델피아반도체나스닥',
 '381170': 'TIGER 미국테크TOP10 INDXX',
 '273130': 'KODEX 종합채권(AA-이상)액티브',
 '148020': 'RISE 200',
 '360200': 'ACE 미국S&P500',
 '458730': 'TIGER 미국배당다우존스',
 '0043B0': 'TIGER 머니마켓액티브',
 '367380': 'ACE 미국나스닥100',
 '481050': 'KODEX CD1년금리플러스액티브(합성)',
 '102780': 'KODEX 삼성그룹',
 '161510': 'PLUS 고배당주',
 '455890': 'RISE 머니마켓액티브',
 '292150': 'TIGER 코리아TOP10',
 '449170': 'TIGER KOFR금리액티브(합성)',
 '395160': 'KODEX AI반도체',
 '487240': 'K

In [ ]:
tickers_info = {
    "069500": {"category": "국내주식", "expense_ratio": 0.15, "dividend_yield": 1.8,
               "keywords": ["코스피", "대형주", "인덱스", "분산투자", "국내주식"]},
    "379800": {"category": "해외주식", "expense_ratio": 0.05, "dividend_yield": 0.0,
               "keywords": ["미국", "S&P500", "대형주", "성장", "해외주식"]},
    "411060": {"category": "배당",     "expense_ratio": 0.01, "dividend_yield": 3.5,
               "keywords": ["미국", "배당", "배당성장", "저비용", "안정"]},
    "305540": {"category": "테마",     "expense_ratio": 0.45, "dividend_yield": 0.0,
               "keywords": ["2차전지", "배터리", "테마", "성장", "고위험"]},
    "379810": {"category": "해외주식", "expense_ratio": 0.07, "dividend_yield": 0.0,
               "keywords": ["미국", "나스닥", "기술주", "성장", "IT"]},
    "381180": {"category": "해외주식", "expense_ratio": 0.49, "dividend_yield": 0.0,
               "keywords": ["반도체", "AI", "미국", "기술주", "고위험"]},
}

In [ ]:
# 맥락을 뽑아낸 후 vectorstore에 저장하여 RAG를 구현할 것

In [ ]:
for t, info in tickers_info.items():
  name = name_by_ticker.get(t, "(이름 미확인)")
  print(f"{t} | {name:30s} | {info['category']}")

069500 | KODEX 200                      | 국내주식
379800 | KODEX 미국S&P500                 | 해외주식
411060 | ACE KRX금현물                     | 배당
305540 | TIGER 2차전지테마                   | 테마
379810 | KODEX 미국나스닥100                 | 해외주식
381180 | TIGER 미국필라델피아반도체나스닥            | 해외주식


### 2. 지표 계산 — `compute_metrics` · `risk_from_vol`

- `compute_metrics(df)` — 연환산 수익률·변동성·MDD 일괄 산출
- `risk_from_vol(vol)` — 변동성 구간을 "낮음 / 약간 낮음 / 중간 / 높음" 등급으로 매핑


In [ ]:
# 티커(번호)를 입력하여, fdr.DataReader()를 이용해 주가 정보의 df를 반환하는 함수 작성
def collect_etf_price(ticker, days=365):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)

    start_str = start_date.strftime('%Y-%m-%d')
    end_str = end_date.strftime('%Y-%m-%d')

    df = fdr.DataReader(ticker, start_str, end_str)

    return {'status' : 'real', 'data': df, 'ticker' : ticker}

collect_etf_price('069500')

{'status': 'real',
 'data':              Open   High    Low  Close    Volume    Change
 Date                                                      
 2025-04-16  32556  32614  32169  32183   7177604 -0.015570
 2025-04-17  32271  32561  32203  32501   6951765  0.009881
 2025-04-18  32556  32722  32452  32722   6025639  0.006800
 2025-04-21  32727  32918  32634  32766   5020620  0.001345
 2025-04-22  32604  32824  32594  32658   5779669 -0.003296
 ...           ...    ...    ...    ...       ...       ...
 2026-04-10  88795  89805  88520  88655  12112242  0.016103
 2026-04-13  86465  88090  86410  87880  13658534 -0.008742
 2026-04-14  90245  91600  89955  90430  17031936  0.029017
 2026-04-15  93405  94180  92205  92665  22556415  0.024715
 2026-04-16  93460  94755  93035  94710  15401829  0.022069
 
 [244 rows x 6 columns],
 'ticker': '069500'}

In [ ]:
def compute_metrics(df):
    close = df['Close'].ffill()
    ret = close.pct_change().dropna()
    ann_ret = ((1 + ret.mean()) ** 252 -1)*100
    ann_vol = ret.std() * np.sqrt(252) * 100
    cum = (1 + ret).cumprod()
    mdd = ((cum - cum.cummax()) / cum.cummax()).min() * 100

    return {'return_1y': round(ann_ret, 2), 'volatility': round(ann_vol, 2), 'mdd' : round(mdd, 2)}

In [ ]:
def risk_from_vol(vol):
  if vol < 5 : return "낮음"
  if vol < 15 : return "약간 낮음"
  if vol < 25 : return "중간"
  return "높음"

In [ ]:
tickers_info

{'069500': {'category': '국내주식',
  'expense_ratio': 0.15,
  'dividend_yield': 1.8,
  'keywords': ['코스피', '대형주', '인덱스', '분산투자', '국내주식']},
 '379800': {'category': '해외주식',
  'expense_ratio': 0.05,
  'dividend_yield': 0.0,
  'keywords': ['미국', 'S&P500', '대형주', '성장', '해외주식']},
 '411060': {'category': '배당',
  'expense_ratio': 0.01,
  'dividend_yield': 3.5,
  'keywords': ['미국', '배당', '배당성장', '저비용', '안정']},
 '305540': {'category': '테마',
  'expense_ratio': 0.45,
  'dividend_yield': 0.0,
  'keywords': ['2차전지', '배터리', '테마', '성장', '고위험']},
 '379810': {'category': '해외주식',
  'expense_ratio': 0.07,
  'dividend_yield': 0.0,
  'keywords': ['미국', '나스닥', '기술주', '성장', 'IT']},
 '381180': {'category': '해외주식',
  'expense_ratio': 0.49,
  'dividend_yield': 0.0,
  'keywords': ['반도체', 'AI', '미국', '기술주', '고위험']}}

In [ ]:
etf_data =[]
for t, info in tickers_info.items():
    result = collect_etf_price(t, days=365)
    # {'status' : 'real', 'data' : df, 'ticker' : ticker}
    metrics = compute_metrics(result['data'])
    name = name_by_ticker.get(t, f"ETF_{t}")
    etf_data.append({
        "ticker" : t,
        "name" : name,
        **info,
        **metrics,
        'risk_level' : risk_from_vol(metrics['volatility']),
        'data_staus' : result['status']
    })

In [ ]:
etf_data

[{'ticker': '069500',
  'name': 'KODEX 200',
  'category': '국내주식',
  'expense_ratio': 0.15,
  'dividend_yield': 1.8,
  'keywords': ['코스피', '대형주', '인덱스', '분산투자', '국내주식'],
  'return_1y': np.float64(227.44),
  'volatility': np.float64(36.58),
  'mdd': -20.62,
  'risk_level': '높음',
  'data_staus': 'real'},
 {'ticker': '379800',
  'name': 'KODEX 미국S&P500',
  'category': '해외주식',
  'expense_ratio': 0.05,
  'dividend_yield': 0.0,
  'keywords': ['미국', 'S&P500', '대형주', '성장', '해외주식'],
  'return_1y': np.float64(40.68),
  'volatility': np.float64(13.33),
  'mdd': -5.7,
  'risk_level': '약간 낮음',
  'data_staus': 'real'},
 {'ticker': '411060',
  'name': 'ACE KRX금현물',
  'category': '배당',
  'expense_ratio': 0.01,
  'dividend_yield': 3.5,
  'keywords': ['미국', '배당', '배당성장', '저비용', '안정'],
  'return_1y': np.float64(60.02),
  'volatility': np.float64(32.95),
  'mdd': -22.76,
  'risk_level': '높음',
  'data_staus': 'real'},
 {'ticker': '305540',
  'name': 'TIGER 2차전지테마',
  'category': '테마',
  'expense_ratio': 0.

### 3. LLM 기반 Description 생성 & Document 구성

- `generate_description(etf)` — 키워드·카테고리를 자연어 설명으로 확장 (LLM)
- 단순 키워드 나열 대신 서술형 문장을 `page_content` 로 사용 → 벡터 검색 품질 향상
- `metadata` 에는 티커·보수율·수익률 등 구조화된 필드 저장 (필터링용)


In [ ]:
# 데이터를 LLM을 활용해 생성 (description 생성)
  # 단순히 keyword를 가져오면 리스트일 테니까, 리스트를 연결해주는 '.'.join()을 사용
def generate_description(etf):

    prompt = f"""다음 ETF의 특징을 한국어 1~2문장으로 간결히 설명하세요.
    이름 : {etf['name']}
    카테고리 : {etf['category']}
    키워드 : {','.join(etf['keywords'])}
    수수료 : {etf['expense_ratio']}% / 배당수익률: {etf['dividend_yield']}$
    1년수익률 : {etf['return_1y']}% / 변동성: {etf['volatility']}% / MDDD : {etf['mdd']}%
    """

    return llm.invoke([{'role' : 'user', 'content' : prompt}]).content.strip()

In [ ]:
documents = []
for etf in etf_data:
  desc = generate_description(etf)
  text = (f"{etf['name']} ({etf['category']}): {desc}"
        f"키워드: {'.'.join(etf['keywords'])}"
        f"수수료: {etf['expense_ratio']}%, 배당수익률:{etf['dividend_yield']}%"
        f"수익률: {etf['return_1y']}% / 변동성: {etf['volatility']}% / MDDD : {etf['mdd']}%")

  metadata = {k:v for k, v in etf.items() if k!='keywords'}
  metadata['keywords'] = '.'.join(etf['keywords'])

  documents.append(Document(page_content=text, metadata = metadata))

In [ ]:
generate_description(etf_data[0])

'KODEX 200은 코스피 대형주에 투자하는 국내주식 ETF로, 분산투자를 통해 안정성을 추구하며, 1년 수익률이 227.44%로 높은 성과를 기록했습니다. 수수료는 0.15%이며, 배당수익률은 1.8%입니다.'

In [ ]:
documents

[Document(metadata={'ticker': '069500', 'name': 'KODEX 200', 'category': '국내주식', 'expense_ratio': 0.15, 'dividend_yield': 1.8, 'return_1y': np.float64(227.44), 'volatility': np.float64(36.58), 'mdd': -20.62, 'risk_level': '높음', 'data_staus': 'real', 'keywords': '코스피.대형주.인덱스.분산투자.국내주식'}, page_content='KODEX 200 (국내주식): KODEX 200은 코스피 대형주에 투자하는 국내주식 인덱스 ETF로, 분산투자를 통해 안정성을 추구합니다. 1년 수익률이 227.44%로 높은 성과를 기록했으며, 수수료는 0.15%입니다.키워드: 코스피.대형주.인덱스.분산투자.국내주식수수료: 0.15%, 배당수익률:1.8%수익률: 227.44% / 변동성: 36.58% / MDDD : -20.62%'),
 Document(metadata={'ticker': '379800', 'name': 'KODEX 미국S&P500', 'category': '해외주식', 'expense_ratio': 0.05, 'dividend_yield': 0.0, 'return_1y': np.float64(40.68), 'volatility': np.float64(13.33), 'mdd': -5.7, 'risk_level': '약간 낮음', 'data_staus': 'real', 'keywords': '미국.S&P500.대형주.성장.해외주식'}, page_content='KODEX 미국S&P500 (해외주식): KODEX 미국S&P500 ETF는 미국 S&P 500 지수를 추종하는 해외주식 ETF로, 대형 성장주에 투자하며 수수료는 0.05%입니다. 최근 1년간 40.68%의 수익률을 기록했으며, 변동성은 13.33%로 MDD는 -5.7%입니다.키워드: 미국.S&P500.대

In [ ]:
# Document 객체를 Vectorstore에 저장

### 4. FAISS Vectorstore 구축

- `LangchainFAISS.from_documents(documents, embeddings)` — 문서 임베딩 + 인덱스 생성 일괄
- `vectorstore.similarity_search_with_score(query, k)` 로 유사도 검색


In [ ]:
vectorstore = LangchainFAISS.from_documents(documents, embeddings)
vectorstore.index.ntotal

6

In [ ]:
query = '미국 기술주에 투자하고 싶어요'
vectorstore.similarity_search_with_score(query, k=3)

[(Document(id='238a15cf-0c81-44e1-bb4c-00179fe0a76b', metadata={'ticker': '381180', 'name': 'TIGER 미국필라델피아반도체나스닥', 'category': '해외주식', 'expense_ratio': 0.49, 'dividend_yield': 0.0, 'return_1y': np.float64(171.26), 'volatility': np.float64(31.35), 'mdd': -10.35, 'risk_level': '높음', 'data_staus': 'real', 'keywords': '반도체.AI.미국.기술주.고위험'}, page_content='TIGER 미국필라델피아반도체나스닥 (해외주식): TIGER 미국필라델피아반도체나스닥 ETF는 미국의 반도체 및 AI 관련 기술주에 투자하는 고위험 상품으로, 최근 1년 동안 171.26%의 높은 수익률을 기록했습니다. 수수료는 0.49%이며, 배당수익률은 0%로 변동성이 31.35%로 나타났습니다.키워드: 반도체.AI.미국.기술주.고위험수수료: 0.49%, 배당수익률:0.0%수익률: 171.26% / 변동성: 31.35% / MDDD : -10.35%'),
  np.float32(1.1617936)),
 (Document(id='c9e25a77-718a-4a25-b0e3-3759fefb7b33', metadata={'ticker': '379810', 'name': 'KODEX 미국나스닥100', 'category': '해외주식', 'expense_ratio': 0.07, 'dividend_yield': 0.0, 'return_1y': np.float64(52.41), 'volatility': np.float64(16.57), 'mdd': -7.34, 'risk_level': '중간', 'data_staus': 'real', 'keywords': '미국.나스닥.기술주.성장.IT'}, page_content='KODEX 미국나스닥100 (해외주

### 5. ETF Knowledge Base & 쿼리 합성

평가·실험용 synthetic query 생성: LLM 에게 `etf_knowledge_base` 전체를 보여주고 가상의 투자자 질문 5개를 만들게 함.


In [ ]:
# ETF knowledge base..
etf_knowledge_base = [
    {"ticker": "069500", "name": "KODEX 200", "category": "국내주식",
     "description": "KOSPI 200 지수 추적. 수수료 0.15%. 대형주 중심 분산투자.",
     "risk": "중간", "expense_ratio": 0.15},
    {"ticker": "379800", "name": "KODEX 미국S&P500TR", "category": "해외주식",
     "description": "S&P500 추적, 배당 자동 재투자. 수수료 0.05%.",
     "risk": "중간", "expense_ratio": 0.05},
    {"ticker": "461460", "name": "KODEX 미국나스닥100TR", "category": "해외주식",
     "description": "나스닥100 기술주 중심. 높은 성장성/변동성. 수수료 0.05%.",
     "risk": "높음", "expense_ratio": 0.05},
]
# knowledge base를 저장했을때 문장들을 뽑아올 수 있도록 하고싶음

In [ ]:
etf_names = [e['name'] + ':' + e['description'] for e in etf_knowledge_base]
etf_names

['KODEX 200:KOSPI 200 지수 추적. 수수료 0.15%. 대형주 중심 분산투자.',
 'KODEX 미국S&P500TR:S&P500 추적, 배당 자동 재투자. 수수료 0.05%.',
 'KODEX 미국나스닥100TR:나스닥100 기술주 중심. 높은 성장성/변동성. 수수료 0.05%.']

In [ ]:
prompt = f"""다음 ETF 데이터를 보고 실제 투자자가 물어볼 법한 질문 5개를 생성하세요

ETF 목록:
{json.dumps(etf_names, ensure_ascii=False, indent = 2)}

형식 : 번호, 질문"""

synthetic = llm.invoke(prompt)

In [ ]:
synthetic

AIMessage(content='1. KODEX 200 ETF는 어떤 종류의 주식에 투자하나요?  \n2. KODEX 미국S&P500TR ETF의 배당금은 어떻게 처리되나요?  \n3. KODEX 미국나스닥100TR ETF의 높은 변동성은 어떤 리스크를 동반하나요?  \n4. KODEX 200 ETF와 KODEX 미국S&P500TR ETF 중 어떤 것이 더 안정적인 투자처인가요?  \n5. 각 ETF의 수수료가 투자 수익에 미치는 영향은 어떻게 되나요?  ', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 122, 'prompt_tokens': 143, 'total_tokens': 265, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DVFsaGQgBl6YgFIJ0VlMTVmx8AjsS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9635-7824-7fb2-add5-3929ef3d8c86-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 143, 'output_tokens': 122, 'total_tokens': 

In [ ]:
# 옵티멀한 값을 찾아야함
  # 청크 사이즈 작으면 -> recall 올라가고
  # 청크 사이즈가 크면 -> 검색을 제대로 못할 수 있음

  # 설명(desc) 간결하게 작성 -> 유사한 것들을 바로 찾을 수 있음
    # 반면 유사한 것의 기준이 모호할 수 있음
  # 길게 작성 -> 답변 풍부할 수 있으나.. 짧은 것과의 trade-off 존재

## BM25 검색 (Kiwi 토크나이저)

한국어 형태소 분석기 기반 키워드 검색.

- `kiwipiepy` 로 형태소 단위 토큰화 (명사·영문·숫자 중심)
- `rank_bm25.BM25Okapi` 로 전 문서 corpus 인덱싱
- 고유명사(예: "KODEX 200")·숫자 조건 질의에 강점


In [ ]:
!pip install -q kiwipiepy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 68.9 MB/s eta 0:00:00


In [ ]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

kiwi = Kiwi()

In [ ]:

def kiwi_tokenize(text):
  tokens = kiwi.tokenize(text)
  result = []
  for token in tokens:
    if token.tag in ("SL", "SN", "NNG", "NNP"):
      result.append(token.form.lower())

  return result

In [ ]:
kiwi_tokenize("KODEX 200 ETF는 어떤 종류의 주식에 투자하나요? ")

['kodex', '200', 'etf', '종류', '주식', '투자']

In [ ]:
kiwi.tokenize("KODEX 200 ETF는 어떤 종류의 주식에 투자하나요? 대형주 중심이라고 했는데 구체적으로 어떤 기업들이 포함되어 있나요?")

[Token(form='KODEX', tag='SL', start=0, len=5),
 Token(form='200', tag='SN', start=6, len=3),
 Token(form='ETF', tag='SL', start=10, len=3),
 Token(form='는', tag='JX', start=13, len=1),
 Token(form='어떤', tag='MM', start=15, len=2),
 Token(form='종류', tag='NNG', start=18, len=2),
 Token(form='의', tag='JKG', start=20, len=1),
 Token(form='주식', tag='NNG', start=22, len=2),
 Token(form='에', tag='JKB', start=24, len=1),
 Token(form='투자', tag='NNG', start=26, len=2),
 Token(form='하', tag='XSV', start=28, len=1),
 Token(form='나요', tag='EF', start=29, len=2),
 Token(form='?', tag='SF', start=31, len=1),
 Token(form='대형주', tag='NNG', start=33, len=3),
 Token(form='중심', tag='NNG', start=37, len=2),
 Token(form='이', tag='VCP', start=39, len=1),
 Token(form='라고', tag='EC', start=40, len=2),
 Token(form='하', tag='VV', start=43, len=1),
 Token(form='었', tag='EP', start=43, len=1),
 Token(form='는데', tag='EC', start=44, len=2),
 Token(form='구체', tag='NNG', start=47, len=2),
 Token(form='적', tag='XSN', 

In [ ]:
test_text = "배당수익률 3% 이상인 미국 ETF를 추천해주세요"
kiwi_tokenize(test_text)

['배당', '수익', '3', '이상', '미국', 'etf', '추천']

In [ ]:
# documents에 문서들을 쌓았고,,,
# 토크나이저를 쌓은 문서들에 이용해서 corpus를 만들 것

corpus = [kiwi_tokenize(doc.page_content) for doc in documents]
corpus

[['kodex',
  '200',
  '국내',
  '주식',
  'kodex',
  '200',
  '코스피',
  '대형주',
  '투자',
  '국내',
  '주식',
  '인덱스',
  'etf',
  '분산',
  '투자',
  '안정',
  '추구',
  '1',
  '수익',
  '227.44',
  '성과',
  '기록',
  '수수료',
  '0.15',
  '키워드',
  '코스피',
  '대형주',
  '인덱스',
  '분산',
  '투자',
  '국내',
  '주식',
  '수수료',
  '0.15',
  '배당',
  '수익',
  '1.8',
  '수익',
  '227.44',
  '변동',
  '36.58',
  'mddd',
  '20.62'],
 ['kodex',
  '미국',
  's',
  'p',
  '500',
  '해외',
  '주식',
  'kodex',
  '미국',
  's',
  'p',
  '500',
  'etf',
  '미국',
  's',
  'p',
  '500',
  '지수',
  '추종',
  '해외',
  '주식',
  'etf',
  '대형',
  '성장',
  '주',
  '투자',
  '수수료',
  '0.05',
  '최근',
  '1',
  '40.68',
  '수익',
  '기록',
  '변동',
  '13.33',
  'mdd',
  '5.7',
  '키워드',
  '미국',
  's',
  'p',
  '500',
  '대형주',
  '성장',
  '해외',
  '주식',
  '수수료',
  '0.05',
  '배당',
  '수익',
  '0.0',
  '수익',
  '40.68',
  '변동',
  '13.33',
  'mddd',
  '5.7'],
 ['ace',
  'krx',
  '금',
  '현물',
  '배당',
  'ace',
  'krx',
  '금',
  '현물',
  'etf',
  '미국',
  '배당',
  '성장',
  '주식',
  '투자',
  '비용',
 

In [ ]:
# bm25 활용

bm25 = BM25Okapi(corpus)

### BM25 검색 함수 & Vector/BM25 비교

- `bm25_search(query, k=5)` — 쿼리 토큰화 → BM25 점수 → Top-K 문서 반환
- 동일 쿼리에 대해 벡터 검색 결과와 BM25 결과를 나란히 출력하여 의미 기반 vs. 키워드 기반 차이 확인


In [ ]:
def bm25_search(query, k=5):
  tokens = kiwi_tokenize(query)
  scores = bm25.get_scores(tokens)
  top_idx = np.argsort(scores)[::-1][:k]

  return [(documents[i], scores[i]) for i in top_idx if scores[i] > 0]


In [ ]:
tokens = kiwi_tokenize('KODEX 200 ETF는 어떤 종류의 주식에 투자하나요?')
scores = bm25.get_scores(tokens)
scores

array([2.63808139, 0.59461242, 0.45602259, 0.36068505, 0.58539451,
       0.44803128])

In [ ]:
top_idx = np.argsort(scores)[::-1][5]
documents[top_idx]

Document(metadata={'ticker': '305540', 'name': 'TIGER 2차전지테마', 'category': '테마', 'expense_ratio': 0.45, 'dividend_yield': 0.0, 'return_1y': np.float64(97.71), 'volatility': np.float64(47.42), 'mdd': -23.19, 'risk_level': '높음', 'data_staus': 'real', 'keywords': '2차전지.배터리.테마.성장.고위험'}, page_content='TIGER 2차전지테마 (테마): TIGER 2차전지테마 ETF는 2차전지 관련 기업에 투자하는 고위험 성장형 테마 ETF로, 최근 1년간 97.71%의 높은 수익률을 기록했으나 변동성이 47.42%로 상당히 크고, 최대 낙폭(MDD)은 -23.19%입니다. 수수료는 0.45%이며 배당수익률은 0%입니다.키워드: 2차전지.배터리.테마.성장.고위험수수료: 0.45%, 배당수익률:0.0%수익률: 97.71% / 변동성: 47.42% / MDDD : -23.19%')

In [ ]:
test_queries = [
    "KODEX 200",
    "안전한 투자 상품",
    "배당 ETF",
    "인플레이션 방어"

]

In [ ]:
for q in test_queries:
  v_result = vectorstore.similarity_search(q, k=1) # vector search 결과
  b_result = bm25_search(q, k=1) # bm25 search 결과
  v_name = v_result[0].metadata['name'] if v_result else "-"
  b_name = b_result[0][0].metadata['name'] if b_result else "-"
  print(f"{q:20s} | {v_name:25s}, {b_name:25s}")

KODEX 200            | KODEX 200                , KODEX 200                
안전한 투자 상품            | TIGER 2차전지테마             , TIGER 미국필라델피아반도체나스닥      
배당 ETF               | ACE KRX금현물               , ACE KRX금현물               
인플레이션 방어             | KODEX 미국나스닥100           , -                        


In [ ]:
# bm25: TF-IDF 의 개선 버전

### BM25 파라미터 튜닝 (k1, b)

- `k1`: 단어 빈도 포화 계수 (기본 1.5). 클수록 빈도 영향 큼
- `b`: 문서 길이 정규화 (기본 0.75). 1에 가까울수록 긴 문서 페널티 강함
- 후보 조합을 돌려 쿼리별 Top-N 변동 관찰


In [ ]:
# k1, b

In [ ]:
query = "배당 ETF"
tokens = kiwi_tokenize(query)
params = [{0.5, 0.25}, (1.5, 0.75), (2.5, 0.9)]
for k1, b in params:
  bm25_temp = BM25Okapi(corpus,k1=k1, b=b)
  scores = bm25_temp.get_scores(tokens)
  top3 = np.argsort(scores)[::-1][:3]
  print(f"k1 = {k1}, b= {b}")
  for idx in top3:
    name = documents[idx].metadata['name']
    print(f"[{scores[idx]}] {name}")


k1 = 0.5, b= 0.25
[0.3613566899426274] TIGER 2차전지테마
[0.36007776374006967] ACE KRX금현물
[0.33230450649179216] KODEX 미국나스닥100
k1 = 1.5, b= 0.75
[0.4545237523360207] ACE KRX금현물
[0.42527693304441117] TIGER 2차전지테마
[0.36870102219246603] KODEX 미국나스닥100
k1 = 2.5, b= 0.9
[0.5268955148093997] ACE KRX금현물
[0.4596282658383855] TIGER 2차전지테마
[0.3891448140870048] KODEX 미국나스닥100


## 메타데이터 필터링 검색

자연어 의미(벡터) 검색 + 구조화된 조건(메타데이터) 을 함께 적용.

- `filtered_search(vectorstore, query, filters, k, fetch_k)`
- `fetch_k` 만큼 후보 → `filters` 조건(예: `{"category": "해외주식"}`) 으로 걸러 `k` 개 반환
- 보수율·배당률 등 숫자 조건은 `less_than` / `greater_than` 형태로 확장 가능


In [ ]:
# 메타데이터 필터링
def filtered_search(vectorstore, query, filters=None, k = 5, fetch_k = 20): # filters = {"expense_ratio": 0.1, "category": "해외주식"} 등이 예시임
  results = vectorstore.similarity_search_with_score(query, fetch_k) # 20개를 먼저 유사한걸 뽑아옴
  if not filters:
    return results[:k]

  filtered = []
  for doc, score in results:
    match = True
    for key, condition in filters.items():
      val = doc.metadata.get(key) # 메타데이터의 카테고리(key) 및 value

      # condition이 딕셔너리 형태일 때 (예: {"less_than": 0.5})
      if isinstance(condition, dict):
          if 'less_than' in condition:
              if not (val < condition['less_than']):
                  match = False
                  break

          if 'greater_than' in condition:
              if not (val > condition['greater_than']):
                  match = False
                  break

      # condition이 일반 값일 때 (예: "해외주식")
      else:
          if val != condition:
              match = False
              break

      if match:
        filtered.append((doc, score)) # 필터에 속하면 추가


  return filtered[:k]

In [ ]:
results = filtered_search(vectorstore, "저비용 해외 ETF", filters = {"category": "해외주식"}, k=3)
results

[(Document(id='c9e25a77-718a-4a25-b0e3-3759fefb7b33', metadata={'ticker': '379810', 'name': 'KODEX 미국나스닥100', 'category': '해외주식', 'expense_ratio': 0.07, 'dividend_yield': 0.0, 'return_1y': np.float64(52.41), 'volatility': np.float64(16.57), 'mdd': -7.34, 'risk_level': '중간', 'data_staus': 'real', 'keywords': '미국.나스닥.기술주.성장.IT'}, page_content='KODEX 미국나스닥100 (해외주식): KODEX 미국나스닥100 ETF는 미국 나스닥 100 지수에 투자하는 해외주식 ETF로, 주로 기술주와 성장주에 집중되어 있습니다. 낮은 수수료(0.07%)와 높은 1년 수익률(52.41%)을 기록하고 있으며, 변동성은 16.57%로 나타났습니다.키워드: 미국.나스닥.기술주.성장.IT수수료: 0.07%, 배당수익률:0.0%수익률: 52.41% / 변동성: 16.57% / MDDD : -7.34%'),
  np.float32(1.0787978)),
 (Document(id='238a15cf-0c81-44e1-bb4c-00179fe0a76b', metadata={'ticker': '381180', 'name': 'TIGER 미국필라델피아반도체나스닥', 'category': '해외주식', 'expense_ratio': 0.49, 'dividend_yield': 0.0, 'return_1y': np.float64(171.26), 'volatility': np.float64(31.35), 'mdd': -10.35, 'risk_level': '높음', 'data_staus': 'real', 'keywords': '반도체.AI.미국.기술주.고위험'}, page_content='TIGER 미국필라델피아반도체나스닥 (해외주식): 

In [ ]:
# 위의 결과에서 expense_ratio가 0.5보다 크다/작다 등의 필터 추가 가능..

results = filtered_search(vectorstore, "저비용 해외 ETF", filters = {"category": "해외주식", "expense_ratio": {"greater_than": 0.5}}, k=3)
results

[(Document(id='c9e25a77-718a-4a25-b0e3-3759fefb7b33', metadata={'ticker': '379810', 'name': 'KODEX 미국나스닥100', 'category': '해외주식', 'expense_ratio': 0.07, 'dividend_yield': 0.0, 'return_1y': np.float64(52.41), 'volatility': np.float64(16.57), 'mdd': -7.34, 'risk_level': '중간', 'data_staus': 'real', 'keywords': '미국.나스닥.기술주.성장.IT'}, page_content='KODEX 미국나스닥100 (해외주식): KODEX 미국나스닥100 ETF는 미국 나스닥 100 지수에 투자하는 해외주식 ETF로, 주로 기술주와 성장주에 집중되어 있습니다. 낮은 수수료(0.07%)와 높은 1년 수익률(52.41%)을 기록하고 있으며, 변동성은 16.57%로 나타났습니다.키워드: 미국.나스닥.기술주.성장.IT수수료: 0.07%, 배당수익률:0.0%수익률: 52.41% / 변동성: 16.57% / MDDD : -7.34%'),
  np.float32(1.0782638)),
 (Document(id='238a15cf-0c81-44e1-bb4c-00179fe0a76b', metadata={'ticker': '381180', 'name': 'TIGER 미국필라델피아반도체나스닥', 'category': '해외주식', 'expense_ratio': 0.49, 'dividend_yield': 0.0, 'return_1y': np.float64(171.26), 'volatility': np.float64(31.35), 'mdd': -10.35, 'risk_level': '높음', 'data_staus': 'real', 'keywords': '반도체.AI.미국.기술주.고위험'}, page_content='TIGER 미국필라델피아반도체나스닥 (해외주식): 